# ALORA vs LoRA Race

**Duration:** ~20-40 min (composes a LoRA checkpoint, embeds the corpus, then runs two vLLM legs back to back; first run also downloads ~6 GB of weights)

**Runtime note:** Each server run takes roughly the same wall time as a real race leg
(3–8 min depending on GPU). The two runs are sequential since Colab typically provides
one GPU; `race_live.html` replays them as if they raced.

This notebook benchmarks two Granite Switch checkpoints — one using **ALORA** (which defers adapter activation to save prefill time) and one using standard **LoRA** — on the same multi-step RAG pipeline, and produces an animated HTML replay of the race. The two servers run sequentially (Colab usually provides one GPU); the replay stitches their telemetry together as if they had raced simultaneously.

*Why vLLM:* the mellea intrinsics API currently supports vLLM only, and the ALORA prefill optimization is implemented in vLLM's Punica kernels.

**What you'll learn:**
- How to run the same pipeline (guardian → query rewrite → retrieval → answerability → clarification → generation) against two Granite Switch checkpoints and produce `race_live.html` + `race_report.html` from the result
- How `--technology-filter lora` lets you compose a like-for-like LoRA-only counterpart to a published ALORA checkpoint
- How to launch, health-check, and tear down vLLM servers from a notebook without leaking GPU memory
- Where ALORA's prefill savings show up in the per-step latency breakdown

**Adapters used:** the embedded ALORA checkpoint [`ibm-granite/granite-switch-4.1-3b-preview`](https://huggingface.co/ibm-granite/granite-switch-4.1-3b-preview) and a LoRA-only build of the same three IBM granitelib libraries — [Core](https://huggingface.co/ibm-granite/granitelib-core-r1.0), [RAG](https://huggingface.co/ibm-granite/granitelib-rag-r1.0), and [Guardian](https://huggingface.co/ibm-granite/granitelib-guardian-r1.0) — composed in section 3.

## Prerequisites

1. **GPU runtime.** A100 or better. In Colab: *Runtime → Change runtime type → A100 GPU*.
2. **HuggingFace login** (cell 4) so the `ibm-granite/*` checkpoints can download.
3. **Run cells in order.** Section 0 clones the repo and `cd`s into the race-script directory; later sections assume that working directory.

New to this series? [`04_compose_granite_switch.ipynb`](./04_compose_granite_switch.ipynb) walks through the composer that section 3 calls. Full setup details (GPU sizes, multi-GPU, troubleshooting) are in [`../PREREQUISITES.md`](../PREREQUISITES.md).


## 0 · Install and set up

In [ ]:
# Install granite-switch with tutorial dependencies (includes vLLM backend).
%pip install -q "granite-switch[tutorials]"
%pip install -q mellea chromadb rich tqdm transformers httpx

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # needed to pull ibm-granite models from the Hub

In [ ]:
# stdlib
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

# third-party
import torch
from IPython.display import HTML

RACE_DIR = "granite-switch/tutorials/scripts/comparison/alora_vs_lora_race"
if not os.path.exists(RACE_DIR):
    RACE_DIR = "tutorials/scripts/comparison/alora_vs_lora_race"
os.chdir(RACE_DIR)
RACE_DIR = os.getcwd()  # absolute, stable across later cells
print("Working dir:", RACE_DIR)

if not torch.cuda.is_available():
    raise RuntimeError("GPU required — enable a GPU runtime in Runtime > Change runtime type")

r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("GPU:", r.stdout.strip())

## 1 · Build ChromaDB index

Downloads 49k government-service passages from the IBM mt-rag-benchmark corpus and embeds
them with `granite-embedding-small-english-r2`. Runs once and persists to `govt_chroma/`.
Skip this cell if `govt_chroma/` is already present.

In [ ]:
if os.path.exists("govt_chroma") and os.listdir("govt_chroma"):
    print("govt_chroma already exists \u2014 skipping build")
else:
    print("Building ChromaDB index (downloads ~50 MB corpus, then embeds 49k passages)...")
    !python build_govt_chroma.py

## 2 · ALORA server

Start `granite-switch-4.1-3b-preview` on port 8111 and run the benchmark against it.

In [ ]:
from granite_switch.tutorials.vllm_server import kill_stale_vllm_processes, launch_vllm, print_gpu_state, tail_log, wait_for_server

# Kill any stale vLLM processes from a previous run — they hold GPU memory even
# after a kernel restart and will prevent a new server from launching.
kill_stale_vllm_processes()
print_gpu_state()

In [ ]:
alora_proc = launch_vllm(
    model    = "ibm-granite/granite-switch-4.1-3b-preview",
    port     = 8111,
    log_file = "/content/vllm_alora.log",
)
if not wait_for_server(8111):
    tail_log("/content/vllm_alora.log")

In [ ]:
# Benchmark the ALORA server.
# --no-live disables Rich Live (which floods notebook output with redrawn frames).
# The animated replay comes from race_live.html at the end.
!python bench_pipeline_race.py --mode sequential --server "ALORA (8111)" --no-live -n 16 -c 8 -k 10

In [ ]:
alora_proc.terminate()
alora_proc.wait()
print("ALORA server stopped")

## 3 · Get the LoRA-only model

The ALORA leg above used the pre-composed `ibm-granite/granite-switch-4.1-3b-preview` from the Hub directly. For a fair comparison the LoRA leg needs a like-for-like checkpoint of the **same** adapter libraries (`granitelib-rag-r1.0`, `granitelib-core-r1.0`, `granitelib-guardian-r1.0`), but with every adapter forced to its standard LoRA variant via `--technology-filter lora`.

Compose that checkpoint via [`./04_compose_granite_switch.ipynb`](./04_compose_granite_switch.ipynb) — pass those three libraries with `--technology-filter lora` and `--output /content/granite-switch-lora-only`. The cell below just records that path so the LoRA server (section 4) knows where to load from; point `LORA_MODEL_DIR` somewhere else if you composed to a different directory.

In [ ]:
LORA_MODEL_DIR = "/content/granite-switch-lora-only"

assert os.path.exists(os.path.join(LORA_MODEL_DIR, "adapter_index.json")), (
    f"No composed LoRA-only checkpoint at {LORA_MODEL_DIR}. "
    "Compose one via tutorials/notebooks/04_compose_granite_switch.ipynb "
    "(pass --technology-filter lora and the three granitelib libraries), "
    "then re-run this cell."
)
print(f"Using LoRA-only checkpoint at {LORA_MODEL_DIR}")

## 4 · LoRA server

Start the LoRA-only composed model on port 8112 and run the same benchmark.

In [ ]:
lora_proc = launch_vllm(
    model    = LORA_MODEL_DIR,
    port     = 8112,
    log_file = "/content/vllm_lora.log",
)
if not wait_for_server(8112):
    tail_log("/content/vllm_lora.log")

In [ ]:
!python bench_pipeline_race.py --mode sequential --server "LORA (8112)" --lora-model {LORA_MODEL_DIR} --no-live -n 16 -c 8 -k 10

In [ ]:
lora_proc.terminate()
lora_proc.wait()
print("LoRA server stopped")

## 5 · Merge results into HTML

Both server runs wrote their data to `race_events.json` and `race_results.json`.
This step copies the HTML templates from `sample_run/` and embeds the data so
the replay and report work as standalone files.

In [ ]:
race_dir = Path(".")
sample   = race_dir / "sample_run"

def embed(template_name, json_name, var_name, marker):
    """Copy template from sample_run/ and embed JSON data."""
    src  = sample / template_name
    dst  = race_dir / template_name
    data = json.loads((race_dir / json_name).read_text())
    shutil.copy2(src, dst)
    html = dst.read_text()
    html = re.sub(
        rf"const {var_name} = .*?; // {marker}",
        f"const {var_name} = {json.dumps(data)}; // {marker}",
        html, flags=re.DOTALL,
    )
    dst.write_text(html)
    print(f"  {template_name}: embedded from {json_name}")

embed("race_live.html",   "race_events.json",  "RACE_EVENTS_EMBEDDED", "<<RACE_EVENTS>>")
embed("race_report.html", "race_results.json", "RACE_DATA_EMBEDDED",  "<<RACE_DATA>>")

## 6 · Results

The animated replay shows the two servers as if they had raced simultaneously — timestamps
are relative to each server's own start, so the replay is accurate for each server's
internal dynamics even though they ran back-to-back.

If the output cell is clipped, click the expand arrows on its left edge.

In [ ]:
# Animated race replay — auto-plays at 5×.
# Use the scrubber or speed buttons to explore the telemetry.
HTML(open("race_live.html").read())

In [ ]:
# Static summary: step latency charts, exit distribution, per-conversation wall times.
HTML(open("race_report.html").read())